# Предсказание цен на недвижимость (California Housing)

В этом проекте мы проведём полный цикл анализа данных и машинного обучения:
1. Загрузка и первичный осмотр данных
2. Разведочный анализ (EDA)
3. Предобработка данных
4. Обучение моделей
5. Сравнение результатов
6. Выводы

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Загрузка данных

Используем встроенный датасет California Housing из scikit-learn. Целевая переменная — медианная стоимость дома (в сотнях тысяч долларов).

In [2]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f'Размер датасета: {df.shape}')
print(f'\nПервые 5 строк:')
df.head()

Размер датасета: (20640, 9)

Первые 5 строк:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
print('Пропущенные значения:')
print(df.isnull().sum())
print('\nСтатистическое описание:')
df.describe().round(4).head(3)

Пропущенные значения:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

Статистическое описание:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.0,20640.0,20640.0,20640.0,20640.0,20640.0,20640.0,20640.0,20640.0
mean,3.8707,28.6395,5.4290,1.0967,1425.4767,3.0707,35.6319,-119.5697,2.0686
std,1.8998,12.5856,2.4742,0.4739,1132.4621,10.3860,2.1360,2.0035,1.1540


## 2. Разведочный анализ данных (EDA)

In [4]:
fig, ax = plt.subplots(figsize=(12, 6))
df['MedHouseVal'].hist(bins=50, edgecolor='black', alpha=0.7, ax=ax)
ax.set_title('Распределение целевой переменной (MedHouseVal)', fontsize=14)
ax.set_xlabel('Медианная стоимость дома (×$100,000)')
ax.set_ylabel('Частота')
ax.axvline(df['MedHouseVal'].median(), color='red', linestyle='--', label=f'Медиана: {df["MedHouseVal"].median():.2f}')
ax.legend()
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

<Figure size 1200x600 with 1 Axes>

In [5]:
fig, ax = plt.subplots(figsize=(12, 10))
correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, ax=ax)
ax.set_title('Матрица корреляций', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

<Figure size 1200x1000 with 1 Axes>

In [6]:
fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(df['Longitude'], df['Latitude'], c=df['MedHouseVal'],
                     cmap='viridis', alpha=0.4, s=5)
plt.colorbar(scatter, label='Стоимость (×$100,000)')
ax.set_title('Географическое распределение цен на недвижимость', fontsize=14)
ax.set_xlabel('Долгота')
ax.set_ylabel('Широта')
plt.tight_layout()
plt.savefig('geo_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

<Figure size 1200x600 with 1 Axes>

In [7]:
top_features = ['MedInc', 'AveRooms', 'HouseAge']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['MedHouseVal'], alpha=0.2, s=5)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('MedHouseVal')
    axes[i].set_title(f'{feat} vs Стоимость')

plt.tight_layout()
plt.savefig('feature_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

<Figure size 1500x500 with 3 Axes>

## 3. Предобработка данных

In [8]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Размер обучающей выборки: {X_train.shape}')
print(f'Размер тестовой выборки: {X_test.shape}')

Размер обучающей выборки: (16512, 8)
Размер тестовой выборки: (4128, 8)


## 4. Обучение и сравнение моделей

In [9]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
}

results = {}

for name, model in models.items():
    if name == 'Linear Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2, 'predictions': y_pred}
    
    print(f'\n=== {name} ===')
    print(f'  MAE:  {mae:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  R²:   {r2:.4f}')


=== Linear Regression ===
  MAE:  0.5332
  RMSE: 0.7460
  R²:   0.5757

=== Decision Tree ===
  MAE:  0.4940
  RMSE: 0.7330
  R²:   0.5905

=== Random Forest ===
  MAE:  0.3275
  RMSE: 0.5009
  R²:   0.8087

=== Gradient Boosting ===
  MAE:  0.3708
  RMSE: 0.5396
  R²:   0.7780


In [10]:
results_df = pd.DataFrame({name: {k: v for k, v in metrics.items() if k != 'predictions'}
                           for name, metrics in results.items()}).T
results_df.sort_values('R²', ascending=False)

,MAE,RMSE,R²
Random Forest,0.3275,0.5009,0.8087
Gradient Boosting,0.3708,0.5396,0.7780
Decision Tree,0.4940,0.7330,0.5905
Linear Regression,0.5332,0.7460,0.5757


## 5. Визуализация результатов

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

model_names = list(results.keys())
r2_scores = [results[m]['R²'] for m in model_names]
mae_scores = [results[m]['MAE'] for m in model_names]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

axes[0].barh(model_names, r2_scores, color=colors, edgecolor='black')
axes[0].set_xlabel('R² Score')
axes[0].set_title('Сравнение моделей по R²')
axes[0].set_xlim(0, 1)
for i, v in enumerate(r2_scores):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center')

axes[1].barh(model_names, mae_scores, color=colors, edgecolor='black')
axes[1].set_xlabel('MAE')
axes[1].set_title('Сравнение моделей по MAE')
for i, v in enumerate(mae_scores):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

<Figure size 1200x500 with 2 Axes>

In [12]:
best_model = models['Random Forest']
best_pred = results['Random Forest']['predictions']

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(y_test, best_pred, alpha=0.3, s=10, color='#3498db')
ax.plot([0, 5.5], [0, 5.5], 'r--', linewidth=2, label='Идеальное предсказание')
ax.set_xlabel('Реальная стоимость')
ax.set_ylabel('Предсказанная стоимость')
ax.set_title('Random Forest: Реальные vs Предсказанные значения')
ax.legend()
plt.tight_layout()
plt.savefig('predictions_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

<Figure size 1000x600 with 1 Axes>

In [13]:
feature_importance = best_model.feature_importances_
feature_names = X.columns
sorted_idx = np.argsort(feature_importance)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_names[sorted_idx], feature_importance[sorted_idx], color='#2ecc71', edgecolor='black')
ax.set_xlabel('Важность признака')
ax.set_title('Random Forest — Важность признаков')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

<Figure size 1000x600 with 1 Axes>

In [14]:
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(best_pred, residuals, alpha=0.3, s=10, color='#e74c3c')
axes[0].axhline(y=0, color='black', linestyle='--')
axes[0].set_xlabel('Предсказанные значения')
axes[0].set_ylabel('Остатки')
axes[0].set_title('Анализ остатков')

axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='#9b59b6')
axes[1].set_xlabel('Остатки')
axes[1].set_ylabel('Частота')
axes[1].set_title('Распределение остатков')
axes[1].axvline(x=0, color='red', linestyle='--')

plt.tight_layout()
plt.savefig('residuals_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

<Figure size 1200x500 with 2 Axes>

## 6. Выводы

### Результаты эксперимента

| Модель | MAE | RMSE | R² |
|---|---|---|---|
| Linear Regression | 0.5332 | 0.7460 | 0.5757 |
| Decision Tree | 0.4940 | 0.7330 | 0.5905 |
| Random Forest | **0.3275** | **0.5009** | **0.8087** |
| Gradient Boosting | 0.3708 | 0.5396 | 0.7780 |

### Ключевые наблюдения

1. **Random Forest** показал лучший результат (R² = 0.81), объясняя ~81% дисперсии целевой переменной.
2. **Медианный доход (MedInc)** — самый важный признак для предсказания стоимости жилья.
3. Географические координаты (Latitude, Longitude) также вносят значительный вклад, что логично: расположение сильно влияет на цену.
4. Линейная регрессия заметно уступает ансамблевым методам, что указывает на нелинейные зависимости в данных.
5. Анализ остатков показывает, что модель хуже справляется с предсказанием очень дорогого жилья (>$500,000).

### Возможные улучшения

- Feature engineering: создание новых признаков (например, комнат на человека)
- Подбор гиперпараметров через GridSearchCV / RandomizedSearchCV
- Использование более сложных моделей (XGBoost, LightGBM)
- Кросс-валидация для более надёжной оценки